# 📦 Notebook 2 — HMM via Libraries### Market Regime Detection · Probabilistic ML ProjectReproduces the 3-state Gaussian HMM regimes from Notebook 1 using:- **`hmmlearn`** — scikit-learn compatible, industry standard- **`pomegranate`** — modern probabilistic ML libraryWe verify numerical agreement with our from-scratch implementation and compare execution speed.---### References1. Nguyen & Nguyen (2015) — *Hidden Markov Model for Stock Selection*, Risks 3(4)  2. McGreevy, J. (2021) — *Hidden Markov Models in Finance*, Imperial College MSc  3. QuantStart — *Market Regime Detection Using HMMs in QSTrader*  4. Chen, X. (2025) — *HMM-based market regime detection with RL*, IDS---

## 0. Imports

In [ ]:
%pip install -q hmmlearn pomegranate

In [ ]:
import sys, os, timesys.path.insert(0, os.path.abspath(".."))import numpy as npimport pandas as pdimport plotly.graph_objects as gofrom plotly.subplots import make_subplotsfrom scipy.stats import norm, wasserstein_distanceimport warningswarnings.filterwarnings("ignore")from hmmlearn import hmm as hmmlearn_hmmtry:    import pomegranate as pg    from pomegranate.hmm import DenseHMM    from pomegranate.distributions import Normal    HAS_POMEGRANATE = True    print(f"pomegranate version: {pg.__version__}")except ImportError:    HAS_POMEGRANATE = False    print("pomegranate not available — skipping that section")from utils.data_utils import prepare_ticker_data, get_observation_sequence, time_splitfrom utils.viz_utils  import (plot_price_with_regimes, plot_regime_distributions,                               plot_transition_matrix, plot_gamma, plot_log_likelihood,                               plot_aic_bic)from utils.metrics    import regime_statistics, aic, bic, hmm_n_paramsfrom hmm_core import GaussianHMM   # our scratch impl — for comparisonprint("\n✅  All imports OK")

## 1. Load Data (same as NB1)

In [ ]:
TICKER = "SPY"df = prepare_ticker_data(TICKER, start="2005-01-01")df_train, df_test = time_split(df, train_frac=0.80)X_train = get_observation_sequence(df_train).reshape(-1, 1)  # hmmlearn needs (T,1)X_test  = get_observation_sequence(df_test).reshape(-1, 1)X_1d    = X_train.ravel()   # 1-D for our scratch implprint(f"Train: {len(X_train)}  Test: {len(X_test)}")

## 2. `hmmlearn` — GaussianHMM`hmmlearn` implements the same Baum-Welch EM internally, using the same scaledforward-backward recursions we built from scratch.

In [ ]:
N_STATES = 3N_ITERS  = 300t0 = time.perf_counter()model_hl = hmmlearn_hmm.GaussianHMM(    n_components=N_STATES,    covariance_type="full",    n_iter=N_ITERS,    tol=1e-7,    random_state=42,    verbose=False,)model_hl.fit(X_train)t_hl = time.perf_counter() - t0print(f"hmmlearn training time : {t_hl:.3f}s")print(f"Converged              : {model_hl.monitor_.converged}")print(f"Log-likelihood (train) : {model_hl.score(X_train):.2f}")print()print("Initial state probs π  :", model_hl.startprob_.round(4))print("Transition matrix A:")print(pd.DataFrame(model_hl.transmat_.round(4)))print("\nMeans µ  :", model_hl.means_.ravel().round(6))print("Std devs σ:", np.sqrt(model_hl.covars_.ravel()).round(6))

In [ ]:
# Decode with hmmlearn (Viterbi)labels_hl = model_hl.predict(X_train)# Sort states low-vol → high-vol for consistent labellingstds_hl   = np.sqrt(model_hl.covars_.ravel())order_hl  = np.argsort(stds_hl)remap_hl  = {order_hl[k]: k for k in range(N_STATES)}labels_hl = np.array([remap_hl[l] for l in labels_hl])state_names = ["Low-Vol", "Med-Vol", "High-Vol"]df_hl = df_train.copy()df_hl["HMM_HL"] = labels_hlfig = plot_price_with_regimes(df_hl, "HMM_HL",                               title="SPY — hmmlearn 3-State HMM Regimes")fig.show()

In [ ]:
fig = plot_regime_distributions(df_hl, "HMM_HL",                                 title="hmmlearn — Regime Return Distributions")fig.show()A_hl = model_hl.transmat_[np.ix_(order_hl, order_hl)]fig  = plot_transition_matrix(A_hl, state_names=state_names,                               title="hmmlearn — Transition Matrix")fig.show()

## 3. From-Scratch vs hmmlearn — Numerical ComparisonWe compare the learned parameters side-by-side.

In [ ]:
# Train scratch implementationt0 = time.perf_counter()hmm_scratch = GaussianHMM(n_states=3, n_iter=300, tol=1e-7, random_state=42)hmm_scratch.fit(X_1d)t_scratch = time.perf_counter() - t0print(f"Scratch training time  : {t_scratch:.3f}s")print(f"hmmlearn training time : {t_hl:.3f}s")print(f"Speed-up (hmmlearn/scratch): {t_scratch/t_hl:.1f}×\n")print("         µ (scratch)   µ (hmmlearn)   σ (scratch)   σ (hmmlearn)")for j in range(3):    print(f"  State {j}: {hmm_scratch.mu_[j]:+.6f}    "          f"{model_hl.means_[order_hl[j],0]:+.6f}      "          f"{hmm_scratch.sigma_[j]:.6f}     "          f"{stds_hl[order_hl[j]]:.6f}")ll_scratch = hmm_scratch.log_likelihood(X_1d)ll_hl      = model_hl.score(X_train) * len(X_train)   # .score returns per-obs LLprint(f"\nLog-likelihood  scratch : {ll_scratch:.2f}")print(f"Log-likelihood hmmlearn : {ll_hl:.2f}")print(f"Difference              : {abs(ll_scratch - ll_hl):.2f}")

In [ ]:
# Regime agreement metriclabels_scratch = hmm_scratch.predict(X_1d)agreement = np.mean(labels_scratch == labels_hl) * 100print(f"State assignment agreement: {agreement:.1f}%  (may differ due to local optima/ordering)")# Regime statistics comparisonprint("\n--- Scratch implementation ---")print(regime_statistics(X_1d, labels_scratch).round(4).to_string(index=False))print("\n--- hmmlearn ---")print(regime_statistics(X_1d, labels_hl).round(4).to_string(index=False))

## 4. pomegranate — Modern Probabilistic HMM`pomegranate` offers PyTorch-backed HMMs with gradient-based options.If not installed, this section is skipped gracefully.

In [ ]:
if HAS_POMEGRANATE:    import torch    X_tensor = torch.tensor(X_train, dtype=torch.float32)    dists = [Normal() for _ in range(N_STATES)]    model_pg = DenseHMM(distributions=dists, max_iter=N_ITERS, tol=1e-7, verbose=False)    t0 = time.perf_counter()    model_pg.fit([X_tensor])    t_pg = time.perf_counter() - t0    labels_pg = model_pg.predict(X_tensor).numpy()    # Sort by std for consistency    pg_means = np.array([d.means.item() for d in model_pg.distributions])    pg_stds  = np.array([d.scales.item() for d in model_pg.distributions])    order_pg = np.argsort(pg_stds)    remap_pg = {order_pg[k]: k for k in range(N_STATES)}    labels_pg = np.array([remap_pg[l] for l in labels_pg])    df_pg = df_train.copy()    df_pg["HMM_PG"] = labels_pg    print(f"pomegranate training time: {t_pg:.3f}s")    print(f"µ: {pg_means[order_pg].round(6)}")    print(f"σ: {pg_stds[order_pg].round(6)}")    fig = plot_price_with_regimes(df_pg, "HMM_PG",                                   title="SPY — pomegranate HMM Regimes")    fig.show()else:    print("Skipping pomegranate — not installed.")

## 5. Model Selection with hmmlearn (N = 2 … 6)

In [ ]:
selection_rows = []for N in range(2, 7):    h = hmmlearn_hmm.GaussianHMM(n_components=N, covariance_type="full",                                   n_iter=300, tol=1e-7, random_state=42)    h.fit(X_train)    ll_tr = h.score(X_train) * len(X_train)    ll_te = h.score(X_test)  * len(X_test)    k     = hmm_n_params(N)    selection_rows.append({        "N": N,        "Train LL": round(ll_tr, 1),        "Test LL":  round(ll_te, 1),        "AIC":      round(aic(ll_tr, k), 1),        "BIC":      round(bic(ll_tr, k, len(X_train)), 1),    })    print(f"N={N}  TrainLL={ll_tr:.1f}  TestLL={ll_te:.1f}  AIC={aic(ll_tr,k):.1f}  BIC={bic(ll_tr,k,len(X_train)):.1f}")sel_df = pd.DataFrame(selection_rows)print("\n", sel_df.to_string(index=False))fig = plot_aic_bic([f"{r['N']}-state" for r in selection_rows],                   [r["AIC"] for r in selection_rows],                   [r["BIC"] for r in selection_rows],                   title="hmmlearn — Model Selection AIC vs BIC")fig.show()

## 6. QSTrader-style Regime SignalThe [QuantStart article](https://www.quantstart.com/articles/market-regime-detection-using-hidden-markov-models-in-qstrader/)uses HMM regime detection as a trading filter. We replicate the logic:- Fit HMM on a rolling window- Use the most recent regime as a directional signal

In [ ]:
# Rolling 252-day HMM regime signalwindow  = 252signals = np.full(len(df), np.nan)X_all_1d = df["Returns"].valuesprint("Computing rolling HMM signals …")for end in range(window, len(df)):    start = end - window    x_win = X_all_1d[start:end].reshape(-1, 1)    h = hmmlearn_hmm.GaussianHMM(n_components=2, covariance_type="full",                                   n_iter=50, tol=1e-4, random_state=0)    h.fit(x_win)    last_state = h.predict(x_win)[-1]    # Low-vol state = bullish signal (1), high-vol = bearish (0)    state_vols = np.sqrt(h.covars_.ravel())    signals[end] = 1 if last_state == np.argmin(state_vols) else 0df_sig = df.copy()df_sig["HMM_Signal"] = signals# Regime-filtered strategy returndf_sig["Strategy"] = df_sig["Returns"] * df_sig["HMM_Signal"].fillna(0)df_sig["BuyHold"]  = df_sig["Returns"]# Equity curveseq_strat = (1 + df_sig["Strategy"]).cumprod()eq_bh    = (1 + df_sig["BuyHold"]).cumprod()fig = go.Figure()fig.add_trace(go.Scatter(x=df_sig.index, y=eq_bh.values,    name="Buy & Hold", line=dict(color="rgba(150,180,255,0.8)", width=1.5)))fig.add_trace(go.Scatter(x=df_sig.index, y=eq_strat.values,    name="HMM-Filtered", line=dict(color="rgba(99,200,180,0.9)", width=1.8)))fig.update_layout(title="Rolling 2-State HMM Regime Filter — Equity Curves",                  yaxis_title="Growth of $1",                  height=450,                  plot_bgcolor="rgba(15,17,22,1)", paper_bgcolor="rgba(15,17,22,1)",                  font=dict(color="#e0e0e0"))fig.update_xaxes(showgrid=True, gridcolor="rgba(80,80,80,0.3)")fig.update_yaxes(showgrid=True, gridcolor="rgba(80,80,80,0.3)")fig.show()strat_sharpe = (df_sig["Strategy"].mean() / df_sig["Strategy"].std()) * np.sqrt(252)bh_sharpe    = (df_sig["BuyHold"].mean()  / df_sig["BuyHold"].std())  * np.sqrt(252)print(f"\nSharpe — Buy & Hold    : {bh_sharpe:.3f}")print(f"Sharpe — HMM-Filtered  : {strat_sharpe:.3f}")print("(Past performance does not predict future returns)")

## 7. Summary| Method | Library | Key Advantage ||---|---|---|| `GaussianHMM` (scratch) | None | Full mathematical transparency || `hmmlearn.GaussianHMM` | hmmlearn | Fast, sklearn-compatible, production-ready || `DenseHMM` | pomegranate | PyTorch backend, GPU-able, flexible distributions |**Next:** `03_alternative_methods.ipynb` — GMM, K-Means, DBSCAN, Random Forest, Wasserstein clustering.